# Bayesian Polynomial Regression

In the previous exercise you used a train/validation split to select the degree of a polynomial regression model and observed the effect of overfitting. In this exercise you will take a **Bayesian** approach, which provides a principled way to:

1. **Regularize** the model by placing a prior distribution over the parameters.
2. **Quantify uncertainty** in predictions via the posterior predictive distribution.
3. **Select the model complexity** (polynomial degree) using the Bayesian Information Criterion (BIC) — without needing a separate validation set.

---

## Background

We use the same polynomial basis functions as before, $\phi_j(x) = x^j$, so the **design matrix** $\boldsymbol{\Phi} \in \mathbb{R}^{n \times (d+1)}$ has entries $\Phi_{ij} = x_i^j$:

$$\boldsymbol{\Phi} = \begin{pmatrix} 1 & x_1 & x_1^2 & \cdots & x_1^d \\ 1 & x_2 & x_2^2 & \cdots & x_2^d \\ \vdots & \vdots & \vdots & \ddots & \vdots \\ 1 & x_n & x_n^2 & \cdots & x_n^d \end{pmatrix}.$$

### Prior

In contrast to last week (linear regression), we treat our model parameters not as constants that need to be found, but instead as random variables.

As a consequence, we can talk about their distribution and define a zero-mean isotropic Gaussian prior on the parameters:
$$\boldsymbol{\beta} \sim \mathcal{N}\bigl(\boldsymbol{0},\; \tau^2 \mathbf{I}\bigr).$$

### Likelihood

Observations are Gaussian around the polynomial prediction:
$$y \mid \boldsymbol{\Phi}, \boldsymbol{\beta} \sim \mathcal{N}\bigl(\boldsymbol{\Phi}\boldsymbol{\beta},\; \sigma^2 \mathbf{I}\bigr).$$

### Posterior

Applying Bayes' theorem yields a Gaussian posterior $p(\boldsymbol{\beta} \mid \mathcal{D}) = \mathcal{N}(\boldsymbol{\beta} \mid \hat{\boldsymbol{\beta}},\, \boldsymbol{\Sigma})$, where

$$\hat{\boldsymbol{\beta}} = \Bigl(\boldsymbol{\Phi}^\top \boldsymbol{\Phi} + \tfrac{\sigma^2}{\tau^2}\mathbf{I}\Bigr)^{-1} \boldsymbol{\Phi}^\top y,
\qquad
\boldsymbol{\Sigma} = \Bigl(\tfrac{1}{\sigma^2}\boldsymbol{\Phi}^\top\boldsymbol{\Phi} + \tfrac{1}{\tau^2}\mathbf{I}\Bigr)^{-1}.$$

### Posterior Predictive

Integrating out $\boldsymbol{\beta}$ at a new input $x^*$ gives

$$p(y^* \mid x^*, \mathcal{D}) = \mathcal{N}\!\Bigl(y^* \mid \boldsymbol{\phi}(x^*)^\top \hat{\boldsymbol{\beta}},\;
\sigma^2 + \boldsymbol{\phi}(x^*)^\top \boldsymbol{\Sigma}\, \boldsymbol{\phi}(x^*)\Bigr).$$

### Bayesian Information Criterion (BIC)

To choose the polynomial degree $d$ without a validation set, we maximize the BIC:

$$\operatorname{BIC}(d) = \sum_{i=1}^n \log \mathcal{N}\!\bigl(y_i \mid \boldsymbol{\phi}(x_i)^\top \hat{\boldsymbol{\beta}},\; \sigma^2\bigr)
- \frac{d+1}{2}\log n,$$

where $d+1$ is the number of model parameters (degrees of freedom).


In [1]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

## Generating Data

The true data-generating function is $y = f(x) = x^3 - x + \eta$, with $\eta$ being Gaussian noise. Here, we draw 6 points and visualize them.

In [2]:
def data_generator(n, noise_std=0.4, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.uniform(-1.5, 1.5, n)
    y = X**3 - X + rng.normal(0, noise_std, n)
    return X, y


X, y = data_generator(n=6)

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=X,
        y=y,
        mode="markers",
        name="Observations",
        marker=dict(symbol="circle-open", size=10),
    )
)
fig.update_layout(
    title="Generated Data", xaxis_title="x", yaxis_title="y", width=800, height=400
)
fig.show()

## Task 1: Design Matrix

Build the polynomial design matrix $\boldsymbol{\Phi}$ for inputs `X` and polynomial degree `d`. The result should be an $(n \times (d+1))$ matrix where column $j$ contains $x_i^j$.

__*Hint:*__ Feel free to reuse the approach from last week's exercise.

In [3]:
def design_matrix(X, d):
    """Return the polynomial design matrix of degree d for inputs X.

    Parameters
    ----------
    X : array of shape (n,)
    d : int, polynomial degree

    Returns
    -------
    Phi : array of shape (n, d+1)
    """
    n = len(X)
    Phi = np.zeros((n, d + 1))

    for i in range(n):
      for j in range(d+1):
        Phi[i,j] = X[i]**j

    return Phi


# Quick check: for degree 2 and X = [0, 1, 2], Phi should be
# [[1, 0, 0],
#  [1, 1, 1],
#  [1, 2, 4]]
print(design_matrix(np.array([0.0, 1.0, 2.0]), d=2))

[[1. 0. 0.]
 [1. 1. 1.]
 [1. 2. 4.]]


## Task 2: Posterior

Implement `posterior`, which computes the posterior mean $\hat{\boldsymbol{\beta}}$ and covariance $\boldsymbol{\Sigma}$ from the formulas in the background section.

__*Hint:*__ Use `np.linalg.inv` for matrix inversion, `@` for matrix multiplication and `np.eye` to create an identity matrix.

In [10]:
def posterior(Phi, y, sigma_squared, tau_squared):
    """Compute the Gaussian posterior over the polynomial coefficients.

    Parameters
    ----------
    Phi           : array of shape (n, d+1), design matrix
    y             : array of shape (n,), observations
    sigma_squared : float, noise variance
    tau_squared   : float, prior variance

    Returns
    -------
    beta_hat : array of shape (d+1,), posterior mean
    Sigma    : array of shape (d+1, d+1), posterior covariance
    """
    n, d_plus_1 = Phi.shape #anzahl reihen und spalten der design matrix
    I = np.eye(d_plus_1) #einheitsmatrix
    Prior = (sigma_squared/tau_squared) * I

    Des_mat = Phi.T @ Phi

    beta_hat = np.linalg.inv(Des_mat + Prior) @ Phi.T @ y

    Sigma = np.linalg.inv((1/sigma_squared) * Phi.T @ Phi + (1/tau_squared)*I)

    return beta_hat, Sigma

## Task 3: Posterior Predictive

Implement `posterior_predictive`, which evaluates the posterior predictive mean and standard deviation at a set of test points (given via their design matrix `Phi`).

For each row $\boldsymbol{\phi}_i$ of `Phi`, the predictive mean and variance are
$$\mu_i = \boldsymbol{\phi}_i^\top \hat{\boldsymbol{\beta}},
\qquad
v_i = \sigma^2 + \boldsymbol{\phi}_i^\top \boldsymbol{\Sigma}\, \boldsymbol{\phi}_i.$$

__*Hint:*__ For the variance you need the diagonal of $\boldsymbol{\Phi} \boldsymbol{\Sigma} \boldsymbol{\Phi}^\top$; you can use `np.diag` to extract it.


In [11]:
def posterior_predictive(Phi, beta_hat, Sigma, sigma_squared):
    """Evaluate the posterior predictive distribution.

    Parameters
    ----------
    Phi           : array of shape (m, d+1), design matrix for prediction points
    beta_hat      : array of shape (d+1,), posterior mean
    Sigma         : array of shape (d+1, d+1), posterior covariance
    sigma_squared : float, noise variance

    Returns
    -------
    y_mean : array of shape (m,), predictive mean
    y_std  : array of shape (m,), predictive standard deviation
    """
    n, d_plus_1 = Phi.shape

    y_mean = np.zeros(n)
    y_std = np.zeros(n)

    for i in range(n):
      y_mean[i] = Phi[i, :] @ beta_hat
      y_var = sigma_squared + Phi[i, :] @ Sigma @ Phi[i, :]
      y_std[i] = np.sqrt(y_var)

    return y_mean, y_std

## Task 4: Model Selection via BIC

Implement `model_selection`, which searches over polynomial degrees $d = 0, 1, \ldots, \texttt{d\_max}$ and returns the degree that maximizes the BIC.

**_Hints:_**
- `norm.logpdf(y, loc=mu, scale=std)` (from `scipy.stats`) gives element-wise logarithm of the multivariate normal distribution; `np.sum` can also be helpful
- Loop over the polynomial degrees
- Use your `design_matrix` and `posterior` functions inside the loop.
- Use `np.argmax` to find the index of the best score; note that the index directly equals the polynomial degree when you iterate from $d=0$.


In [17]:
import scipy.stats

def model_selection(X, y, sigma_squared, tau_squared, d_max=15):
    """Select the polynomial degree that maximizes the BIC.

    Parameters
    ----------
    X, y          : training inputs and observations
    sigma_squared : float, noise variance
    tau_squared   : float, prior variance
    d_max         : int, maximum degree to consider

    Returns
    -------
    best_d : int, the degree with the highest BIC score
    """
    n = len(X)
    bic_scores = []
    degrees = np.arange(d_max + 1)

    for d in degrees:
        # 1. Design Matrix für den aktuellen Grad d erstellen
        # (Ich nehme an, deine Funktion heißt design_matrix)
        Phi = design_matrix(X, d)

        # 2. Posterior berechnen, um beta_hat zu erhalten
        beta_hat, _ = posterior(Phi, y, sigma_squared, tau_squared)

        # 3. Log-Likelihood berechnen: sum(log p(y | Phi, beta_hat, sigma_squared))
        # Der Mittelwert der Vorhersage ist Phi @ beta_hat
        y_pred_mean = Phi @ beta_hat

        # Wir nutzen scipy, um die Log-Dichte der Normalverteilung für alle Punkte zu summieren
        log_likelihood = np.sum(scipy.stats.norm.logpdf(y, loc=y_pred_mean, scale=np.sqrt(sigma_squared)))

        # 4. BIC berechnen: Log-Likelihood - (Anzahl Parameter / 2) * log(n)
        # Anzahl Parameter ist d + 1
        current_bic = log_likelihood - ((d + 1) / 2) * np.log(n)

        bic_scores.append(current_bic)

    # 5. Den Grad mit dem höchsten BIC-Score finden
    best_d = degrees[np.argmax(bic_scores)]

    return int(best_d)

## Plotting helper
You do not need to read this part.

In [18]:
def make_frame(n, X_plot, X_n, y_n, y_mean, y_std, title):
    """Build one Plotly Frame for observation count n."""
    y_true = X_plot**3 - X_plot
    return go.Frame(
        data=[
            go.Scatter(
                x=X_plot,
                y=y_true,
                mode="lines",
                name="True function",
                line=dict(color="blue"),
            ),
            go.Scatter(
                x=X_plot,
                y=y_mean,
                mode="lines",
                name="Posterior predictive mean",
                line=dict(color="red"),
            ),
            go.Scatter(
                x=np.concatenate([X_plot, X_plot[::-1]]),
                y=np.concatenate([y_mean + y_std, (y_mean - y_std)[::-1]]),
                fill="toself",
                fillcolor="rgba(255, 0, 0, 0.2)",
                line=dict(color="rgba(255, 255, 255, 0)"),
                name="±1 std",
            ),
            go.Scatter(
                x=X_n,
                y=y_n,
                mode="markers",
                name="Observations",
                marker=dict(color="black", symbol="circle-open", size=10),
            ),
        ],
        layout=go.Layout(title_text=title),
        name=str(n),
    )


def make_slider_fig(frames, titles):
    """Assemble a Plotly Figure with a slider from pre-built frames."""
    slider_steps = [
        {
            "args": [
                [str(n)],
                {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"},
            ],
            "label": str(n),
            "method": "animate",
        }
        for n in range(1, len(frames) + 1)
    ]
    return go.Figure(
        data=frames[0].data,
        frames=frames,
        layout=go.Layout(
            template="plotly",
            title=titles[0],
            xaxis_title="x",
            yaxis_title="y",
            width=800,
            height=550,
            yaxis_range=[-2.5, 3.0],
            sliders=[
                {
                    "active": 0,
                    "steps": slider_steps,
                    "x": 0.1,
                    "len": 0.8,
                    "xanchor": "left",
                    "yanchor": "top",
                    "pad": {"b": 10, "t": 50},
                    "currentvalue": {
                        "prefix": "n = ",
                        "visible": True,
                        "xanchor": "center",
                    },
                }
            ],
        ),
    )

## Putting It All Together

We pre-generate **15 fixed observations** of $y = f(x) = x^3 - x + \eta$. Use the slider to step from $n = 1$ to $n = 15$ observations and watch how:

- BIC selects the best polynomial degree for the current $n$,
- the **posterior predictive mean** (red) gets closer to the true function (blue),
- the **±1 std band** (shaded) shrinks as more data arrives.

Feel free to experiment with the parameters, e.g. $\sigma^2$ and $\tau^2$, $n$, or the data-generating function.


In [19]:
sigma_squared = 0.4 ** 2
tau_squared = 5.0 ** 2

X_all, y_all = data_generator(n=15)
X_plot = np.linspace(-1.5, 1.5, 200)

frames, titles = [], []

for n in range(1, len(X_all) + 1):
    X_n, y_n = X_all[:n], y_all[:n]

    d_best = model_selection(X_n, y_n, sigma_squared, tau_squared)
    Phi = design_matrix(X_n, d_best)
    beta_hat, Sigma = posterior(Phi, y_n, sigma_squared, tau_squared)
    Phi_plot = design_matrix(X_plot, d_best)
    y_mean, y_std = posterior_predictive(Phi_plot, beta_hat, Sigma, sigma_squared)

    title = f"Bayesian Polynomial Regression (degree {d_best}, n={n})"
    titles.append(title)
    frames.append(make_frame(n, X_plot, X_n, y_n, y_mean, y_std, title))

fig = make_slider_fig(frames, titles)
fig.show()